# How to manually create a simulation
In this notebook we deeply understand the `sat_com_topology` library by creating a simulation **without any configuration**, object by object, using `SimulationManager` directly. We connect the ISS (International Space Station) to a Ground Station through a User Terminal:

```
User Terminal --- ISS --- Ground Station
```

This is the lowest level way of building a simulation. If you only want to run an existing constellation shape, [create_my_first_simulation.ipynb](create_my_first_simulation.ipynb) is a better starting point.

In [ ]:
from datetime import datetime
from pathlib import Path

from sat_com_adapter.adapters import NetworkXAdapter
from sat_com_application.time_managers import SimulationTimeManager
from sat_com_builder.configuration_manager import EmptyConfigurationManager
from sat_com_model.models import create_satellite, create_ground_station, create_user_terminal
from sat_com_trajectopy.orbital_models import PyOrbitalModel


## Generate an empty simulation
`EmptyConfigurationManager` gives you a `SimulationManager` with no satellite, no ground station and no user terminal. Everything below is added by hand.

In [ ]:
empty_configuration_manager = EmptyConfigurationManager()

simulation_manager = empty_configuration_manager.load_simulation()


## Configure a simulation clock
Every orbital computation needs a clock to know "when" to compute a satellite position. This is the role of the `SimulationTimeManager`: it holds the current simulation time and drives every object that depends on it (movement models, links updates, ...).

We register it on the `simulation_manager` right away, since the satellite's movement model will need it in the next step.

The TLE we use below has an epoch of `2014-09-30`. SGP4 (the propagation model behind `pyorbital`) only stays accurate for a few weeks around its TLE's epoch, so we pick a simulation date close to it. Picking a date far from the epoch (e.g. today) would raise a `"Satellite crashed"` error, since the propagated orbit becomes physically invalid. In a real project you would instead fetch a TLE whose epoch is close to the date you actually want to simulate.

In [ ]:
date_string = "30/09/2014 14:00:00"
date_format = "%d/%m/%Y %H:%M:%S"

start_date = datetime.strptime(date_string, date_format)
end_date = start_date

time_manager = SimulationTimeManager(start_date=start_date, end_date=end_date)
simulation_manager.set_time_manager(time_manager)


## Configure CESIUM TOKEN
We will use it later to render our simulation in 3D.

In [ ]:
CESIUM_TOKEN = "<YOUR_CESIUM_TOKEN>"


## Create the satellite (ISS)
### Define the TLE
A [Two-Line Element set](https://en.wikipedia.org/wiki/Two-line_element_set) (TLE) describes an orbit. Here is a TLE for the ISS.

In [ ]:
tle = {
    "satellite_name": "ISS (ZARYA)",
    "line1": "1 25544U 98067A   14273.50403866  .00012237  00000-0  21631-3 0  1790",
    "line2": "2 25544  51.6467 297.5710 0002045 126.1182  27.2142 15.50748592907666",
}


### Create the satellite

In [ ]:
satellite = create_satellite(tle["satellite_name"], 0)


### Set the Movement Model
`PyOrbitalModel` computes the satellite position from its TLE. It needs the simulation's `TimeManager` to know at which time to compute the position, so it always stays in sync with the rest of the simulation.

In [ ]:
orbital_model = PyOrbitalModel(tle, simulation_manager.time_manager)
satellite.set_movement_model(orbital_model)


### Add the Satellite to the simulation

In [ ]:
simulation_manager.add_satellite(satellite)


## Configure Ground Station

In [ ]:
ground_station = create_ground_station(1)
ground_station.set_position(
    longitude=2.349014,
    latitude=48.864716,
    altitude=0,
)
simulation_manager.add_ground_station(ground_station)


## Configure User Terminal

In [ ]:
user_terminal = create_user_terminal(0, "wanna_connect")
user_terminal.set_position(
    longitude=1.444000,
    latitude=43.604500,
    altitude=0,
)
simulation_manager.add_user_terminals(user_terminal)


## Create links
Links are created explicitly through the `SimulationManager`, there is no automatic connection strategy involved here since we are not using any `GroundObjectDomain`.

In [ ]:
simulation_manager.create_and_add_ground_station_link_connection(satellite, ground_station)
simulation_manager.create_and_add_user_terminal_link_connection(satellite, user_terminal)


## Bonus: measure the distance to the ISS
Most `SimulationManager` helpers that reason about distances (closest satellite, range based connections, ...) need a `DistanceModel`. Let's set one and measure how far the Ground Station currently is from the ISS.

In [ ]:
from sat_com_trajectopy.distance_models import SkLearningDistanceModel

simulation_manager.simulation.set_distance_model(SkLearningDistanceModel())

distance_km = simulation_manager.get_distance_between_two_topology_object_km(ground_station, satellite)
print(f"Ground distance between Paris and the ISS: {distance_km:.0f} km")


## Export simulation
### Export to NetworkX

In [ ]:
Path("results").mkdir(exist_ok=True)

networkx_adapter = NetworkXAdapter(simulation_manager)
networkx_adapter.create_full_networkx_graph(export_object_position=True)
networkx_adapter.adapt(output_directory="results/deeply_understand-networkx.json")


### Visualize with Cesium
The Cesium renderer styles each object per its "domain" (`WalkerShell` for satellites, `GroundObjectDomain` for ground stations and user terminals) to pick their color. Since every object here was created by hand, none of them has a domain yet, so we attach minimal placeholders just for rendering purposes.

In [ ]:
from sat_com_model.models import WalkerShell, GroundObjectDomain

satellite.walker_shell = WalkerShell(
    identifier="Manual Satellites",
    type="manual",
    amount_of_orbit_plane=1,
    amount_of_satellite_per_orbit_plane=1,
)
ground_station.ground_object_domain = GroundObjectDomain(
    identifier="Manual Ground Stations", type="ground_station"
)
user_terminal.ground_object_domain = GroundObjectDomain(
    identifier="Manual User Terminals", type="user_terminal"
)


> **Note:** `sat_com_adapter==4.2.0` renders a User Terminal *marker* through `user_terminal.user_name`, an attribute that no longer exists on `UserTerminal` (only `label` does), which raises an `AttributeError`. Until this is fixed upstream, we build the Cesium render by hand and skip `add_terminal_users()`; the User Terminal link is still drawn correctly, only the point marker itself is missing.

In [ ]:
from sat_com_adapter.cesium_renderer.models import CesiumProperty
from sat_com_adapter.cesium_renderer.renderers import CesiumHtmlRenderer

cesium_renderer = CesiumHtmlRenderer(simulation_manager, CesiumProperty(cesium_token=CESIUM_TOKEN))
cesium_renderer.add_satellites()
cesium_renderer.add_ground_stations()
cesium_renderer.add_inter_satellite_links()
cesium_renderer.add_ground_station_links()
cesium_renderer.add_user_terminal_links()

with open("results/deeply_understand-cesium.html", "w") as file:
    file.write(cesium_renderer.render())


Open [`results/deeply_understand-cesium.html`](results/deeply_understand-cesium.html) in your browser to see the result.

## Next step
Now that you understand the building blocks, go to [shortest_path_lab.ipynb](shortest_path_lab.ipynb) to compute a shortest path between a User Terminal and a Ground Station.